## Упражнение 04. A/B-тестирование

In [1]:
import pandas as pd
import sqlite3

### 1. Создаем соединение с базой данных с помощью библиотеки sqlite3

In [2]:
conn=sqlite3.connect('../data/checking-logs.sqlite')

### 2. Одним запросом на группу создай два DataFrame: test_results и control_results - с колонками "time" и "avg_diff", и ровно двумя строками.
#### - В "time" должны быть значения "after" и "before";
#### - В "avg_diff" - средняя дельта для всех пользователей за периоды до и после их первого визита на страницу;
#### - Учитывай только тех пользователей, у кого есть наблюдения и «до», и «после».
### 3. Лабораторную project1 по-прежнему не учитываем.


In [3]:
test_query= """
SELECT
CASE
WHEN test.first_commit_ts < test.first_view_ts THEN 'before'
ELSE 'after' END AS time,
AVG((unixepoch(test.first_commit_ts) - deadlines.deadlines) / 3600) AS avg_diff
FROM test
INNER JOIN deadlines ON test.labname = deadlines.labs
WHERE test.labname != 'project1' AND
test.first_commit_ts IS NOT NULL AND
test.first_view_ts IS NOT NULL
GROUP BY time;
"""
test_results = pd.io.sql.read_sql(test_query,conn)
test_results

,time,avg_diff
0,after,-103.40625
1,before,-60.56250


In [4]:
control_query= """
SELECT
CASE
WHEN control.first_commit_ts < control.first_view_ts THEN 'before'
ELSE 'after'
END AS time,
AVG((unixepoch(control.first_commit_ts) - deadlines.deadlines) / 3600) AS avg_diff
FROM control
INNER JOIN deadlines ON control.labname = deadlines.labs
WHERE control.labname != 'project1' AND
control.first_commit_ts IS NOT NULL AND
control.first_view_ts IS NOT NULL
GROUP BY time;
"""
control_results = pd.io.sql.read_sql(control_query,conn)
control_results

,time,avg_diff
0,after,-112.710526
1,before,-99.464286


### 4. Закрываем соединение

In [5]:
conn.close()

### Дай ответ: подтвердилась ли гипотеза - влияет ли страница на поведение студентов?

In [6]:
print("Yes" if (test_results.iloc[:, 1][0] < test_results.iloc[:, 1][1]) and (control_results.iloc[:, 1][0] < control_results.iloc[:, 1][1]) else "No")

Yes
